# Train the crop disease model (Colab GPU, works in the browser or the VS Code Colab extension)

**VS Code:** Select Kernel > Colab > New Colab Server > GPU (T4). The code runs on Colab's machine, not your PC,
so the first cells clone the repo there, and the model is sent back by uploading it to Hugging Face Hub.

**Keys:** in the browser, add them in the 🔑 Secrets panel (`ROBOFLOW_API_KEY`, `HF_TOKEN`, and `GITHUB_TOKEN` if the
repo is private). In VS Code, or if a secret is missing, the cell asks you to paste it instead. Keys are never saved.

Output: `model_<version>/` with `model.tflite`, `classes.json`, `calibration.json`, `preprocess.json`, `metrics.json`
(see `contracts/model_io.md`).

In [ ]:
REPO = "YOUR-USER/YOUR-REPO"            # <- change this
BRANCH = "main"
VERSION = "v0.2"
FINE_TUNE_EPOCHS = 8                    # 0 = heads only (fast); 8 is a good start on a T4
# Loss settings: copy the best run from compare_datasets.py (results.md)
FIELD_WEIGHT, MISS_PENALTY, FOCAL_GAMMA, MAX_MISSED = 3, 1, 2, 0.05
USE_PLANT_PATHOLOGY = False            # True = add Plant Pathology 2021 apple photos (needs a Kaggle token)
HF_MODEL_REPO = "YOUR-HF-USER/crop-disease-model"   # <- private model repo on Hugging Face (made if missing)
!nvidia-smi -L

In [ ]:
import os
from getpass import getpass

def secret(name, required=True):
    """Colab Secrets panel when available (browser), otherwise ask (VS Code)."""
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    value = getpass(f"{name} (leave empty to skip): " if not required else f"{name}: ")
    return value or None

token = secret("GITHUB_TOKEN", required=False)  # only for a private repo
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
!rm -rf /content/repo && git clone -q --depth 1 -b {BRANCH} {url} /content/repo
%cd /content/repo
!pip install -q -r ml/requirements.txt huggingface_hub

## 1. PlantVillage (lab photos: our 13 classes, plus other crops for other_plant)

In [ ]:
!git clone -q --depth 1 --filter=blob:none --sparse https://github.com/spMohanty/PlantVillage-Dataset /content/pv
!cd /content/pv && git sparse-checkout set --no-cone "/raw/color/*"
!python ml/src/prepare_plantvillage.py /content/pv/raw/color /content/data

## 2. Roboflow (field photos)

Fill `ml/configs/roboflow_datasets.json` first (in the repo, then push). Run the class listing, check the mapping, then download.

In [ ]:
key = secret("ROBOFLOW_API_KEY", required=False)  # empty = skip Roboflow (PlantVillage-only model)
if key:
    os.environ["ROBOFLOW_API_KEY"] = key
!python ml/src/download_roboflow.py --list-classes --cache /content/rf

In [ ]:
!python ml/src/download_roboflow.py --out /content/data --cache /content/rf --scenes-out /content/scenes
!for d in /content/data/*/; do echo "$(ls $d | wc -l) $(ls $d | grep -c '^rf_') $(basename $d)"; done  # total, field, label

## 2b. Plant Pathology 2021 (optional, apple orchard photos)

Accept the competition rules on Kaggle first. The cell asks for your Kaggle username and API key.

In [ ]:
if USE_PLANT_PATHOLOGY:
    os.environ["KAGGLE_USERNAME"] = secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = secret("KAGGLE_KEY")
    !pip install -q kaggle
    !kaggle competitions download -c plant-pathology-2021-fgvc8 -p /content
    !unzip -q -o /content/plant-pathology-2021-fgvc8.zip -d /content/pp2021 && rm /content/plant-pathology-2021-fgvc8.zip
    !python ml/src/prepare_plant_pathology.py /content/pp2021 /content/data


## 2c. Compare datasets and loss settings (heads only, a few minutes on GPU)

Pick the run with the best selection score in the table, then set FIELD_WEIGHT / MISS_PENALTY / FOCAL_GAMMA above.

In [ ]:
!python ml/src/compare_datasets.py --data /content/data --out /content/compare


## 3. Train, calibrate, export

In [ ]:
!python ml/src/train.py --data /content/data --out /content/model_{VERSION} --version {VERSION} --fine-tune-epochs {FINE_TUNE_EPOCHS} --field-weight {FIELD_WEIGHT} --miss-penalty {MISS_PENALTY} --focal-gamma {FOCAL_GAMMA} --max-missed {MAX_MISSED}


In [ ]:
import json
m = json.load(open(f"/content/model_{VERSION}/metrics.json"))
for head, s in m["per_head"].items():
    f = s.get("app_rule_field_photos", s["app_rule_all_photos"])
    print(f"{head}: field P {f['macro_precision']} R {f['macro_recall']} F1 {f['macro_f1']} F2 {f['macro_f2']}"
          + (f" | missed {f['missed_disease_rate']} false alarm {f['false_alarm_rate']} tau_h {s['healthy_threshold']}"
             if head != "plant" else ""))
    for lab, v in f["per_class"].items():
        print(f"    {lab:28s} P {v['precision']}  R {v['recall']}  n={v['support']}")


In [ ]:
# Multi-leaf test: whole field photos, whole-photo answer vs leaf tiles (test split only)
!pip install -q -r backend/requirements.txt
!cd backend && python ../ml/src/eval_scenes.py --model-dir /content/model_{VERSION} --scenes /content/scenes --out /content/model_{VERSION}/scenes_eval.json


In [ ]:
# Send the model back: upload to Hugging Face Hub (private). Each VERSION gets its own tag; older tags keep
# their files. splits.json is included so eval_scenes.py works on your PC. Then download it from the repo's "Files" tab,
# or on your PC: huggingface-cli download YOUR-HF-USER/crop-disease-model --revision v0.1 --local-dir backend/models/v0.1
from huggingface_hub import HfApi
api = HfApi(token=secret("HF_TOKEN"))
api.create_repo(HF_MODEL_REPO, private=True, exist_ok=True)
api.upload_folder(folder_path=f"/content/model_{VERSION}", repo_id=HF_MODEL_REPO,
                  commit_message=f"model {VERSION}")
api.create_tag(HF_MODEL_REPO, tag=VERSION, exist_ok=True)
print(f"https://huggingface.co/{HF_MODEL_REPO}/tree/{VERSION}")

In [ ]:
# Browser only (optional): download a zip directly instead.
!cd /content && zip -qr model_{VERSION}.zip model_{VERSION}
from google.colab import files
files.download(f"/content/model_{VERSION}.zip")